# 05 — Semissupervisionado e importância

## Objetivo

Self-training melhora o ranking quando parte dos rótulos é ocultada? E como
interpretar globalmente o modelo escolhido?

A demonstração semissupervisionada é curta. Depois usamos importâncias nativas
e Permutation Importance, sem explicações individuais.

In [1]:
from pathlib import Path
import sys

ponto_atual = Path.cwd().resolve()
RAIZ = next(
    caminho for caminho in (ponto_atual, *ponto_atual.parents)
    if (caminho / "data" / "raw" / "UCI_Credit_Card.csv").exists()
)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


import json
import time
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.semi_supervised import SelfTrainingClassifier

from src.auxiliares import (
    COLUNAS_NOMINAIS,
    PARAMETROS_REFERENCIA,
    avaliar_probabilidades,
    carregar_base_preparada,
    criar_modelo_gradiente,
    separar_dados,
)
from src.visual_utils import grafico_importancia_variaveis

dados = carregar_base_preparada(RAIZ)
X_treino, X_validacao, X_teste, y_treino, y_validacao, y_teste = separar_dados(dados)
caminho_parametros = RAIZ / "models" / "parametros_gradient_boosting.json"
parametros = json.loads(caminho_parametros.read_text()) if caminho_parametros.exists() else PARAMETROS_REFERENCIA.copy()

## Como simular poucos rótulos disponíveis?

In [2]:
posicoes = np.arange(len(X_treino))
posicoes_rotuladas, _ = train_test_split(
    posicoes, train_size=0.25, stratify=y_treino, random_state=42,
)
X_treino_array = X_treino.to_numpy()
X_validacao_array = X_validacao.to_numpy()
y_treino_array = y_treino.to_numpy()
rotulos_parciais = np.full(len(y_treino_array), -1, dtype=int)
rotulos_parciais[posicoes_rotuladas] = y_treino_array[posicoes_rotuladas]

## Qual estimador será usado no self-training?

In [3]:
indices_nominais = [X_treino.columns.get_loc(coluna) for coluna in COLUNAS_NOMINAIS]
indices_numericos = [i for i in range(X_treino.shape[1]) if i not in indices_nominais]

def criar_estimador_base():
    preprocessamento = ColumnTransformer([
        ("nominais", OneHotEncoder(handle_unknown="ignore", sparse_output=False), indices_nominais),
        ("numericas", StandardScaler(), indices_numericos),
    ])
    return Pipeline([
        ("preprocessamento", preprocessamento),
        ("modelo", LogisticRegression(max_iter=2000, random_state=42)),
    ])

## O self-training supera o mesmo subconjunto rotulado?

In [4]:
modelo_supervisionado = criar_estimador_base()
modelo_supervisionado.fit(
    X_treino_array[posicoes_rotuladas],
    y_treino_array[posicoes_rotuladas],
)
probabilidades_supervisionadas = modelo_supervisionado.predict_proba(X_validacao_array)[:, 1]

modelo_self_training = SelfTrainingClassifier(
    estimator=criar_estimador_base(),
    criterion="threshold",
    threshold=0.90,
    max_iter=10,
)
modelo_self_training.fit(X_treino_array, rotulos_parciais)
probabilidades_self_training = modelo_self_training.predict_proba(X_validacao_array)[:, 1]

In [5]:
resultados_semissupervisionado = pd.DataFrame([
    avaliar_probabilidades("Supervisionado — 25% rotulado", y_validacao, probabilidades_supervisionadas),
    avaliar_probabilidades("Self-training", y_validacao, probabilidades_self_training),
])
pseudorrotulos = modelo_self_training.transduction_[rotulos_parciais == -1]
print("Pseudorrótulos adicionados:", int((pseudorrotulos != -1).sum()))
resultados_semissupervisionado

Pseudorrótulos adicionados: 6728


,modelo,limiar,precision,recall,f1,pr_auc,vn,fp,fn,vp,tempo_treino_s
0,Supervisionado — 25% rotulado,0.5,0.720455,0.238885,0.358800,0.494637,4550,123,1010,317,NaN
1,Self-training,0.5,0.694332,0.258478,0.376716,0.490670,4522,151,984,343,NaN


O self-training demonstra a técnica, mas não melhora a AP / PR-AUC neste
experimento. Ele não será a solução final.

## Quais variáveis o modelo usa globalmente?

In [6]:
modelo_interpretado = criar_modelo_gradiente(parametros)
modelo_interpretado.fit(X_treino, y_treino)
nomes_transformados = modelo_interpretado.named_steps[
    "preprocessamento"
].get_feature_names_out()
importancias_nativas = pd.DataFrame({
    "variavel": nomes_transformados,
    "importancia": modelo_interpretado.named_steps["modelo"].feature_importances_,
}).sort_values("importancia", ascending=False)
importancias_nativas.head(12)

,variavel,importancia
15,status_pagamento_set,0.581047
16,status_pagamento_ago,0.070676
21,valor_fatura_set,0.045914
17,status_pagamento_jul,0.042165
13,limite_credito,0.038411
28,valor_pago_ago,0.025482
27,valor_pago_set,0.024578
29,valor_pago_jul,0.023025
14,idade,0.015736
19,status_pagamento_mai,0.015703


In [7]:
fig = grafico_importancia_variaveis(
    importancias_nativas,
    titulo="Importância nativa do Gradient Boosting",
)
fig.show()

## A importância permanece ao embaralhar cada feature original?

In [8]:
inicio = time.perf_counter()
permutacao = permutation_importance(
    modelo_interpretado, X_validacao, y_validacao,
    scoring="average_precision", n_repeats=10,
    random_state=42, n_jobs=-1,
)
tempo_permutacao = time.perf_counter() - inicio
importancias_permutacao = pd.DataFrame({
    "variavel": X_validacao.columns,
    "importancia_media": permutacao.importances_mean,
    "desvio": permutacao.importances_std,
}).sort_values("importancia_media", ascending=False)
print(f"Tempo: {tempo_permutacao:.1f} s")
importancias_permutacao.head(12)

Tempo: 8.7 s


,variavel,importancia_media,desvio
5,status_pagamento_set,0.183604,0.007473
0,limite_credito,0.014703,0.001954
6,status_pagamento_ago,0.013676,0.001114
11,valor_fatura_set,0.012408,0.003714
7,status_pagamento_jul,0.008988,0.002625
19,valor_pago_jul,0.006217,0.001053
8,status_pagamento_jun,0.005179,0.000904
17,valor_pago_set,0.004796,0.001828
10,status_pagamento_abr,0.003702,0.001439
9,status_pagamento_mai,0.003682,0.000966


In [9]:
fig = grafico_importancia_variaveis(
    importancias_permutacao,
    coluna_valor="importancia_media",
    titulo="Permutation Importance na validação",
    coluna_desvio="desvio",
)
fig.show()

## Resultado

O status de pagamento mais recente lidera as duas leituras. Importância
preditiva não implica causalidade, e variáveis mensais correlacionadas podem
dividir importância.